# **From Waveform to Genre — Evaluation**

## Deep Learning Genre Classification

This notebook provides a critical evaluation of the deep learning pipeline developed in `data_prep.ipynb`, `models_train.ipynb`, and `analysis.ipynb` (`notebooks/deep_learning/`). It does not modify or re-run the original training code; it loads the already-generated results and metadata from Google Drive and focuses on questions that were not addressed in the original notebooks: sample size adequacy, data leakage risk, baseline comparisons, and a clearer interpretation of per-class performance and model behavior.

**FMA dataset (citation)**
Defferrard, M., Benzi, K., Vandergheynst, P., & Bresson, X. (2017). *FMA: A Dataset for Music Analysis*. 18th ISMIR. PDF: https://arxiv.org/pdf/1612.01840.pdf

**License / data note:** FMA metadata is licensed under CC BY 4.0; audio files are distributed under per-artist Creative Commons terms. This notebook uses only precomputed metadata, metrics, and evaluation results, and does not redistribute original audio files. Any use of FMA-derived materials should include proper attribution according to original dataset terms.

**Relationship to previous notebooks:** All results referenced here (`inference_results.csv`, `classification_report.json`, `confusion_matrix.png`, `training_history.csv`, `all_misclassifications.csv`, `train.csv`, `test.csv`) are outputs generated by the previous deep learning pipeline (`data_prep.ipynb`, `models_train.ipynb`, and `analysis.ipynb`) and stored in Google Drive under `waveform_analysis_outputs/data_storage`. No model retraining is performed in this notebook.  

**Execution order** Before running this notebook, data_prep.ipynb, models_train.ipynb, and analysis.ipynb must be executed in this order to generate the required datasets and evaluation artifacts. This notebook does not retrain the model; it only loads and analyzes the previously generated results.

## 1. Environment Setup & Artifact Loading

This section connects to Google Drive and loads all precomputed artifacts generated by the deep learning pipeline:
- `train.csv` & `test.csv` — Dataset splits and metadata
- `inference_results.csv` — Test set predictions and ground truth
- `classification_report.json` — Per-class precision, recall, and F1-score
- `training_history.csv` — Loss and accuracy progression per epoch
- `all_misclassifications.csv` — Detailed breakdown of incorrect predictions

In [ ]:
from google.colab import drive
# mount Google Drive to access files stored in it
drive.mount('/content/drive')

In [ ]:
import os
import shutil
import json
import pandas as pd

# path to the original files in Google Drive
DRIVE_SOURCE = '/content/drive/MyDrive/waveform_analysis_outputs/data_storage'

# local working directory in Colab (isolated from Drive)
LOCAL_STORAGE = '/content/evaluation_data'
os.makedirs(LOCAL_STORAGE, exist_ok=True)

# copying the files to Colab
if os.path.exists(DRIVE_SOURCE):
    for filename in os.listdir(DRIVE_SOURCE):
        src_file = os.path.join(DRIVE_SOURCE, filename)
        dst_file = os.path.join(LOCAL_STORAGE, filename)
        if os.path.isfile(src_file):
            shutil.copy2(src_file, dst_file)
            print(f"Copied: {filename}")
else:
    raise FileNotFoundError(f"Directory not found: {DRIVE_SOURCE}")

# the files are loaded into memory only from the local copy in Colab.
train_df = pd.read_csv(os.path.join(LOCAL_STORAGE, 'train.csv'))
test_df = pd.read_csv(os.path.join(LOCAL_STORAGE, 'test.csv'))
inference_df = pd.read_csv(os.path.join(LOCAL_STORAGE, 'inference_results.csv'))
history_df = pd.read_csv(os.path.join(LOCAL_STORAGE, 'training_history.csv'))

with open(os.path.join(LOCAL_STORAGE, 'classification_report.json'), 'r') as f:
    class_report = json.load(f)

print("-" * 50)
print(f"Train samples: {len(train_df)}")
print(f"Test samples:  {len(test_df)}")
print("All data is loaded from the local copy in Colab. The originals in Drive are intact.")

In [ ]:
# Check the exact size and class distribution of all dataset splits

split_dfs = {
    "train": train_df,
    "validation": pd.read_csv(os.path.join(LOCAL_STORAGE, "val.csv")),
    "test": test_df
}

for split_name, df in split_dfs.items():
    print(f"{split_name.capitalize()} samples: {len(df)}")
    print("Columns:", list(df.columns))
    print()

total_samples = sum(len(df) for df in split_dfs.values())
print(f"Total samples across all splits: {total_samples}")